# Area C Processing

This notebook downloads and cleans Comune di Milano Open Data Area C hourly access data by vehicle category.

The useful family for this project is the hourly vehicle-category series for 2012-2016. Area C is a central-Milan exposure proxy: it is most relevant for crash analyses inside the Mura Spagnole / central rings, not for all-city traffic risk.

Raw files are stored in `data/raw/` and processed files are written to `data/processed/`.


In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
import requests

pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.3f}".format

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
project_root = next(root for root in candidate_roots if (root / "data").exists())
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

SOURCES = {
    2012: {
        "page": "https://dati.comune.milano.it/dataset/ds824_area_c_accessi_orari_2012_suddivisi_per_categoria_veicolo",
        "csv_url": "https://dati.comune.milano.it/dataset/d8743377-0b6f-460f-ada4-75ad77a2d709/resource/49dfce7a-459e-490d-8cbf-7ae4c01f9971/download/v2_accessiorari_disaggregazionecategoria_veicoloareacundefined_2012_fix.csv",
        "raw_path": raw_dir / "area_c_accessi_orari_2012_categoria_veicolo.csv",
    },
    2013: {
        "page": "https://dati.comune.milano.it/dataset/ds825_area_c_accessi_orari_2013_suddivisi_per_categoria_veicolo",
        "csv_url": "https://dati.comune.milano.it/dataset/5b6676c0-21ea-483d-954e-d44a975bb885/resource/bf578e80-1996-4844-9355-3b355a1f5329/download/v2_accessiorari_disaggregazionecategoria_veicoloareacundefined_2013.csv",
        "raw_path": raw_dir / "area_c_accessi_orari_2013_categoria_veicolo.csv",
    },
    2014: {
        "page": "https://dati.comune.milano.it/dataset/ds826_area_c_accessi_orari_2014_suddivisi_per_categoria_veicolo",
        "csv_url": "https://dati.comune.milano.it/dataset/fb19b6da-739c-4d2c-96b8-d9139ec6a2e8/resource/4106892a-ae6c-4b59-8308-23d9b11941c6/download/v2_accessiorari_disaggregazionecategoria_veicoloareacundefined_2014_fix.csv",
        "raw_path": raw_dir / "area_c_accessi_orari_2014_categoria_veicolo.csv",
    },
    2015: {
        "page": "https://dati.comune.milano.it/dataset/ds827_area_c_accessi_orari_2015_suddivisi_per_categoria_veicolo",
        "csv_url": "https://dati.comune.milano.it/dataset/93f2c8ca-54f9-416d-b885-845647a6a7d2/resource/f6d76135-5387-4d5b-86fb-73c4878ec79a/download/v2_accessiorari_disaggregazionecategoria_veicoloareacundefined_2015_fix.csv",
        "raw_path": raw_dir / "area_c_accessi_orari_2015_categoria_veicolo.csv",
    },
    2016: {
        "page": "https://dati.comune.milano.it/dataset/ds828_area_c_accessi_orari_2016_suddivisi_per_categoria_veicolo",
        "csv_url": "https://dati.comune.milano.it/dataset/fb89ce04-1522-4582-a134-499d138ace64/resource/95e14237-8050-487e-9370-aa0aeb18daa5/download/v2_accessiorari_disaggregazionecategoria_veicoloareacundefined_2016_fix.csv",
        "raw_path": raw_dir / "area_c_accessi_orari_2016_categoria_veicolo.csv",
    },
}

CENTRAL_RINGS_INSIDE_MURA_SPAGNOLE = [
    "Entro la Cerchia dei Navigli",
    "Dalla Cerchia dei Navigli alle Mura Spagnole",
]

print(f"Project root: {project_root}")


Project root: /Users/faustozamparelli/Developer/MilanCrash


## Download or Manual Load

The portal exposes direct CSV resources for 2012-2016 hourly vehicle-category data.

Manual fallback paths follow this pattern:
- `data/raw/area_c_accessi_orari_YYYY_categoria_veicolo.csv`


In [2]:
def download_if_missing(year, meta):
    path = meta["raw_path"]
    if path.exists():
        print(f"{year}: found {path.relative_to(project_root)}")
        return path

    print(f"{year}: downloading from {meta['csv_url']}")
    response = requests.get(meta["csv_url"], timeout=60)
    response.raise_for_status()
    path.write_bytes(response.content)
    print(f"{year}: saved {path.relative_to(project_root)} ({path.stat().st_size:,} bytes)")
    return path

raw_paths = {year: download_if_missing(year, meta) for year, meta in SOURCES.items()}


2012: found data/raw/area_c_accessi_orari_2012_categoria_veicolo.csv
2013: found data/raw/area_c_accessi_orari_2013_categoria_veicolo.csv
2014: found data/raw/area_c_accessi_orari_2014_categoria_veicolo.csv
2015: found data/raw/area_c_accessi_orari_2015_categoria_veicolo.csv
2016: found data/raw/area_c_accessi_orari_2016_categoria_veicolo.csv


## Schema Inspection

Inspect each raw CSV before any cleaning assumptions are applied.


In [3]:
raw_frames = {}
for year, path in raw_paths.items():
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    raw_frames[year] = df
    print("=" * 80)
    print(f"Year {year}")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Dtypes:")
    print(df.dtypes)
    print("Missing values:")
    print(df.isna().sum())
    display(df.head())


Year 2012
Shape: (17568, 8)
Columns: ['data', 'timeIndex', 'NC', 'ALTRO', 'BUS', 'MERCI', 'PERSONE', 'areac']
Dtypes:
data             str
timeIndex        str
NC           float64
ALTRO        float64
BUS          float64
MERCI        float64
PERSONE      float64
areac        float64
dtype: object
Missing values:
data            0
timeIndex       0
NC            102
ALTRO         115
BUS          1097
MERCI          98
PERSONE        98
areac          98
dtype: int64


,data,timeIndex,NC,ALTRO,BUS,MERCI,PERSONE,areac
0,2012-01-01,00:00,24.000,9.000,15.000,23.000,844.000,0.000
1,2012-01-01,00:30,30.000,7.000,22.000,36.000,1728.000,0.000
2,2012-01-01,01:00,31.000,13.000,22.000,42.000,2249.000,0.000
3,2012-01-01,01:30,44.000,12.000,17.000,43.000,2359.000,0.000
4,2012-01-01,02:00,29.000,8.000,20.000,43.000,2295.000,0.000


Year 2013
Shape: (17520, 8)
Columns: ['data', 'timeIndex', 'NC', 'ALTRO', 'BUS', 'MERCI', 'PERSONE', 'areac']
Dtypes:
data             str
timeIndex        str
NC             int64
ALTRO        float64
BUS          float64
MERCI        float64
PERSONE      float64
areac          int64
dtype: object
Missing values:
data            0
timeIndex       0
NC              0
ALTRO         161
BUS          1134
MERCI          80
PERSONE        80
areac           0
dtype: int64


,data,timeIndex,NC,ALTRO,BUS,MERCI,PERSONE,areac
0,2013-01-01,00:00,110,3.000,11.000,21.000,830.000,0
1,2013-01-01,00:30,115,8.000,15.000,45.000,1595.000,0
2,2013-01-01,01:00,117,11.000,16.000,34.000,2028.000,0
3,2013-01-01,01:30,98,14.000,15.000,36.000,2259.000,0
4,2013-01-01,02:00,137,10.000,21.000,38.000,2158.000,0


Year 2014
Shape: (17520, 8)
Columns: ['data', 'timeIndex', 'NC', 'ALTRO', 'BUS', 'MERCI', 'PERSONE', 'areac']
Dtypes:
data             str
timeIndex        str
NC           float64
ALTRO        float64
BUS          float64
MERCI        float64
PERSONE      float64
areac        float64
dtype: object
Missing values:
data            0
timeIndex       0
NC            768
ALTRO         829
BUS          1782
MERCI         770
PERSONE       770
areac         768
dtype: int64


,data,timeIndex,NC,ALTRO,BUS,MERCI,PERSONE,areac
0,2014-01-01,00:00,189.000,3.000,6.000,10.000,650.000,0.000
1,2014-01-01,00:30,287.000,4.000,21.000,25.000,1191.000,0.000
2,2014-01-01,01:00,375.000,5.000,14.000,37.000,1777.000,0.000
3,2014-01-01,01:30,412.000,8.000,10.000,25.000,1862.000,0.000
4,2014-01-01,02:00,409.000,8.000,10.000,29.000,1694.000,0.000


Year 2015
Shape: (17520, 8)
Columns: ['data', 'timeIndex', 'NC', 'ALTRO', 'BUS', 'MERCI', 'PERSONE', 'areac']
Dtypes:
data             str
timeIndex        str
NC           float64
ALTRO        float64
BUS          float64
MERCI        float64
PERSONE      float64
areac        float64
dtype: object
Missing values:
data            0
timeIndex       0
NC           2546
ALTRO        2591
BUS          2993
MERCI        2548
PERSONE      2548
areac        2546
dtype: int64


,data,timeIndex,NC,ALTRO,BUS,MERCI,PERSONE,areac
0,2015-01-01,00:00,151.000,7.000,8.000,18.000,668.000,0.000
1,2015-01-01,00:30,234.000,6.000,10.000,19.000,1279.000,0.000
2,2015-01-01,01:00,264.000,6.000,8.000,22.000,1647.000,0.000
3,2015-01-01,01:30,325.000,10.000,9.000,35.000,1946.000,0.000
4,2015-01-01,02:00,333.000,9.000,13.000,23.000,1682.000,0.000


Year 2016
Shape: (17568, 8)
Columns: ['data', 'timeIndex', 'NC', 'ALTRO', 'BUS', 'MERCI', 'PERSONE', 'areac']
Dtypes:
data             str
timeIndex        str
NC           float64
ALTRO        float64
BUS          float64
MERCI        float64
PERSONE      float64
areac        float64
dtype: object
Missing values:
data            0
timeIndex       0
NC           7962
ALTRO        7203
BUS          7300
MERCI        6153
PERSONE      6072
areac        6070
dtype: int64


,data,timeIndex,NC,ALTRO,BUS,MERCI,PERSONE,areac
0,2016-01-01,00:00,155.000,5.000,11.000,11.000,636.000,0.000
1,2016-01-01,00:30,301.000,5.000,15.000,13.000,1226.000,0.000
2,2016-01-01,01:00,407.000,10.000,17.000,14.000,1750.000,0.000
3,2016-01-01,01:30,470.000,7.000,16.000,29.000,1813.000,0.000
4,2016-01-01,02:00,411.000,2.000,13.000,13.000,1639.000,0.000


## Cleaning Helpers

Column names are standardized to snake case. Date, time, Area C activity, and vehicle-category access columns are detected from the inspected schema.


In [4]:
def snake_case(name):
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text).strip("_").lower()
    return text


def find_one(columns, keywords, required=True):
    matches = [col for col in columns if any(key in col.lower() for key in keywords)]
    if not matches and required:
        raise ValueError(f"Could not detect column with keywords {keywords}. Available: {list(columns)}")
    return matches[0] if matches else None


def derive_hour(series):
    parsed = pd.to_datetime(series.astype(str), format="%H:%M", errors="coerce")
    if parsed.notna().any():
        return parsed.dt.hour
    extracted = series.astype(str).str.extract(r"(\d{1,2})", expand=False)
    return pd.to_numeric(extracted, errors="coerce")


def assign_time_band(hour):
    if pd.isna(hour):
        return pd.NA
    hour = int(hour)
    if 0 <= hour <= 5:
        return "night"
    if 6 <= hour <= 9:
        return "morning_peak"
    if 10 <= hour <= 15:
        return "midday"
    if 16 <= hour <= 19:
        return "evening_peak"
    if 20 <= hour <= 23:
        return "evening"
    return pd.NA


def clean_area_c_frame(df, source_year):
    working = df.copy()
    working.columns = [snake_case(col) for col in working.columns]

    date_col = find_one(working.columns, ["data", "date", "giorno"])
    hour_col = find_one(working.columns, ["time", "ora", "hour"])
    areac_col = find_one(working.columns, ["areac", "area_c", "area"], required=False)

    reserved = {date_col, hour_col}
    if areac_col:
        reserved.add(areac_col)

    long_category_col = find_one(working.columns, ["categoria", "category", "veicolo"], required=False)
    accesses_col = find_one(working.columns, ["access", "accessi", "ingress", "ingressi", "count", "numero"], required=False)

    if long_category_col and accesses_col and long_category_col != accesses_col:
        long_df = working[[date_col, hour_col, long_category_col, accesses_col] + ([areac_col] if areac_col else [])].copy()
        long_df = long_df.rename(columns={long_category_col: "vehicle_category", accesses_col: "accesses"})
    else:
        category_cols = [
            col for col in working.columns
            if col not in reserved and pd.to_numeric(working[col], errors="coerce").notna().any()
        ]
        if not category_cols:
            raise ValueError(f"No vehicle-category access columns detected for {source_year}")
        long_df = working.melt(
            id_vars=[date_col, hour_col] + ([areac_col] if areac_col else []),
            value_vars=category_cols,
            var_name="vehicle_category",
            value_name="accesses",
        )

    long_df = long_df.rename(columns={date_col: "date", hour_col: "source_time"})
    if areac_col and areac_col in long_df.columns:
        long_df = long_df.rename(columns={areac_col: "areac"})
    else:
        long_df["areac"] = pd.NA

    long_df["date"] = pd.to_datetime(long_df["date"], errors="coerce").dt.date
    long_df["hour"] = derive_hour(long_df["source_time"]).astype("Int64")
    long_df["vehicle_category"] = long_df["vehicle_category"].astype(str).map(snake_case)
    long_df["accesses"] = pd.to_numeric(long_df["accesses"], errors="coerce")
    long_df["areac"] = pd.to_numeric(long_df["areac"], errors="coerce").astype("Int64")

    long_df = (
        long_df
        .dropna(subset=["date", "hour", "vehicle_category", "accesses"])
        .groupby(["date", "hour", "vehicle_category"], as_index=False, dropna=False)
        .agg(accesses=("accesses", "sum"), areac=("areac", "max"))
    )

    date_ts = pd.to_datetime(long_df["date"])
    long_df["year"] = date_ts.dt.year.astype("Int64")
    long_df["month"] = date_ts.dt.month.astype("Int64")
    long_df["day"] = date_ts.dt.day.astype("Int64")
    long_df["weekday"] = date_ts.dt.weekday.astype("Int64")
    long_df["weekday_name"] = date_ts.dt.day_name()
    long_df["is_weekend"] = long_df["weekday"].isin([5, 6])
    long_df["time_band"] = long_df["hour"].map(assign_time_band)

    ordered = [
        "date", "year", "month", "day", "weekday", "weekday_name", "is_weekend",
        "hour", "time_band", "vehicle_category", "accesses", "areac",
    ]
    long_df = long_df[ordered].sort_values(["date", "hour", "vehicle_category"]).reset_index(drop=True)

    detection = {
        "source_year": source_year,
        "date_col": date_col,
        "hour_col": hour_col,
        "areac_col": areac_col,
        "vehicle_category_columns": sorted(long_df["vehicle_category"].unique().tolist()),
    }
    return long_df, detection


## Build Long Hourly Vehicle-Category Data


In [5]:
cleaned_frames = []
detections = []
for year, df in raw_frames.items():
    cleaned, detection = clean_area_c_frame(df, year)
    cleaned_frames.append(cleaned)
    detections.append(detection)
    print("=" * 80)
    print(detection)
    print("Cleaned shape:", cleaned.shape)
    display(cleaned.head())

area_c_long = pd.concat(cleaned_frames, ignore_index=True)
area_c_long = area_c_long.sort_values(["date", "hour", "vehicle_category"]).reset_index(drop=True)

print("Combined shape:", area_c_long.shape)
print("Date range:", area_c_long["date"].min(), "to", area_c_long["date"].max())
print("Vehicle categories:", sorted(area_c_long["vehicle_category"].unique()))
print("Missing values:")
print(area_c_long.isna().sum())


{'source_year': 2012, 'date_col': 'data', 'hour_col': 'timeindex', 'areac_col': 'areac', 'vehicle_category_columns': ['altro', 'bus', 'merci', 'nc', 'persone']}
Cleaned shape: (43400, 12)


,date,year,month,day,weekday,weekday_name,is_weekend,hour,time_band,vehicle_category,accesses,areac
0,2012-01-01,2012,1,1,6,Sunday,True,0,night,altro,16.000,0
1,2012-01-01,2012,1,1,6,Sunday,True,0,night,bus,37.000,0
2,2012-01-01,2012,1,1,6,Sunday,True,0,night,merci,59.000,0
3,2012-01-01,2012,1,1,6,Sunday,True,0,night,nc,54.000,0
4,2012-01-01,2012,1,1,6,Sunday,True,0,night,persone,2572.000,0


{'source_year': 2013, 'date_col': 'data', 'hour_col': 'timeindex', 'areac_col': 'areac', 'vehicle_category_columns': ['altro', 'bus', 'merci', 'nc', 'persone']}
Cleaned shape: (43320, 12)


,date,year,month,day,weekday,weekday_name,is_weekend,hour,time_band,vehicle_category,accesses,areac
0,2013-01-01,2013,1,1,1,Tuesday,False,0,night,altro,11.000,0
1,2013-01-01,2013,1,1,1,Tuesday,False,0,night,bus,26.000,0
2,2013-01-01,2013,1,1,1,Tuesday,False,0,night,merci,66.000,0
3,2013-01-01,2013,1,1,1,Tuesday,False,0,night,nc,225.000,0
4,2013-01-01,2013,1,1,1,Tuesday,False,0,night,persone,2425.000,0


{'source_year': 2014, 'date_col': 'data', 'hour_col': 'timeindex', 'areac_col': 'areac', 'vehicle_category_columns': ['altro', 'bus', 'merci', 'nc', 'persone']}
Cleaned shape: (41576, 12)


...

{'source_year': 2015, 'date_col': 'data', 'hour_col': 'timeindex', 'areac_col': 'areac', 'vehicle_category_columns': ['altro', 'bus', 'merci', 'nc', 'persone']}
Cleaned shape: (37253, 12)


,date,year,month,day,weekday,weekday_name,is_weekend,hour,time_band,vehicle_category,accesses,areac
0,2015-01-01,2015,1,1,3,Thursday,False,0,night,altro,13.000,0
1,2015-01-01,2015,1,1,3,Thursday,False,0,night,bus,18.000,0
2,2015-01-01,2015,1,1,3,Thursday,False,0,night,merci,37.000,0
3,2015-01-01,2015,1,1,3,Thursday,False,0,night,nc,385.000,0
4,2015-01-01,2015,1,1,3,Thursday,False,0,night,persone,1947.000,0


{'source_year': 2016, 'date_col': 'data', 'hour_col': 'timeindex', 'areac_col': 'areac', 'vehicle_category_columns': ['altro', 'bus', 'merci', 'nc', 'persone']}
Cleaned shape: (28296, 12)


,date,year,month,day,weekday,weekday_name,is_weekend,hour,time_band,vehicle_category,accesses,areac
0,2016-01-01,2016,1,1,4,Friday,False,0,night,altro,10.000,0
1,2016-01-01,2016,1,1,4,Friday,False,0,night,bus,26.000,0
2,2016-01-01,2016,1,1,4,Friday,False,0,night,merci,24.000,0
3,2016-01-01,2016,1,1,4,Friday,False,0,night,nc,456.000,0
4,2016-01-01,2016,1,1,4,Friday,False,0,night,persone,1862.000,0


Combined shape: (193845, 12)
Date range: 2012-01-01 to 2016-09-18
Vehicle categories: ['altro', 'bus', 'merci', 'nc', 'persone']
Missing values:
date                0
year                0
month               0
day                 0
weekday             0
weekday_name        0
is_weekend          0
hour                0
time_band           0
vehicle_category    0
accesses            0
areac               0
dtype: int64


## Build Wide Hourly Table


In [6]:
wide_counts = (
    area_c_long
    .pivot_table(index=["date", "hour"], columns="vehicle_category", values="accesses", aggfunc="sum", fill_value=0)
    .reset_index()
)
wide_counts.columns.name = None
wide_counts = wide_counts.rename(columns={cat: f"accesses_{cat}" for cat in wide_counts.columns if cat not in ["date", "hour"]})
wide_counts["total_accesses"] = wide_counts[[c for c in wide_counts.columns if c.startswith("accesses_")]].sum(axis=1)

hour_meta = (
    area_c_long
    .groupby(["date", "hour"], as_index=False)
    .agg(
        areac=("areac", "max"),
        year=("year", "first"),
        month=("month", "first"),
        weekday=("weekday", "first"),
        is_weekend=("is_weekend", "first"),
        time_band=("time_band", "first"),
    )
)
area_c_wide = wide_counts.merge(hour_meta, on=["date", "hour"], how="left")

expected_category_cols = ["accesses_persone", "accesses_merci", "accesses_bus", "accesses_altro", "accesses_nc"]
for col in expected_category_cols:
    if col not in area_c_wide.columns:
        area_c_wide[col] = 0

area_c_wide = area_c_wide[
    ["date", "hour", "total_accesses"]
    + expected_category_cols
    + ["areac", "year", "month", "weekday", "is_weekend", "time_band"]
]
area_c_wide = area_c_wide.sort_values(["date", "hour"]).reset_index(drop=True)

print("Wide shape:", area_c_wide.shape)
display(area_c_wide.head())


Wide shape: (39109, 14)


,date,hour,total_accesses,accesses_persone,accesses_merci,accesses_bus,accesses_altro,accesses_nc,areac,year,month,weekday,is_weekend,time_band
0,2012-01-01,0,2738.000,2572.000,59.000,37.000,16.000,54.000,0,2012,1,6,True,night
1,2012-01-01,1,4832.000,4608.000,85.000,39.000,25.000,75.000,0,2012,1,6,True,night
2,2012-01-01,2,4348.000,4158.000,77.000,36.000,16.000,61.000,0,2012,1,6,True,night
3,2012-01-01,3,3113.000,2939.000,72.000,31.000,22.000,49.000,0,2012,1,6,True,night
4,2012-01-01,4,2195.000,2073.000,56.000,35.000,6.000,25.000,0,2012,1,6,True,night


## Monthly Exposure Table

The main crash datasets are monthly, not hourly. This aggregation is the bridge that makes Area C useful for the existing inference notebooks.


In [7]:
category_cols = [col for col in area_c_wide.columns if col.startswith("accesses_")]
monthly_sum = (
    area_c_wide
    .groupby(["year", "month"], as_index=False)
    .agg(
        month_start=("date", "min"),
        month_end=("date", "max"),
        days_observed=("date", "nunique"),
        hourly_rows=("total_accesses", "size"),
        total_accesses=("total_accesses", "sum"),
        mean_hourly_accesses=("total_accesses", "mean"),
        active_hours=("areac", lambda s: int((s == 1).sum())),
        inactive_hours=("areac", lambda s: int((s == 0).sum())),
    )
)
monthly_categories = area_c_wide.groupby(["year", "month"], as_index=False)[category_cols].sum()
area_c_monthly = monthly_sum.merge(monthly_categories, on=["year", "month"], how="left")
area_c_monthly["areac_active_hour_share"] = area_c_monthly["active_hours"] / area_c_monthly["hourly_rows"]
for col in category_cols:
    area_c_monthly[col.replace("accesses_", "share_")] = area_c_monthly[col] / area_c_monthly["total_accesses"].where(area_c_monthly["total_accesses"] != 0)
area_c_monthly["month_start"] = pd.to_datetime(area_c_monthly["month_start"]).dt.to_period("M").dt.to_timestamp()
area_c_monthly["month_end"] = pd.to_datetime(area_c_monthly["month_end"])

print("Monthly exposure shape:", area_c_monthly.shape)
print("Monthly date range:", area_c_monthly["month_start"].min().date(), "to", area_c_monthly["month_start"].max().date())
display(area_c_monthly.head())


Monthly exposure shape: (56, 21)
Monthly date range: 2012-01-01 to 2016-09-01


,year,month,month_start,month_end,days_observed,hourly_rows,total_accesses,mean_hourly_accesses,active_hours,inactive_hours,accesses_persone,accesses_merci,accesses_bus,accesses_altro,accesses_nc,areac_active_hour_share,share_persone,share_merci,share_bus,share_altro,share_nc
0,2012,1,2012-01-01,2012-01-31,31,744,4053264.000,5447.935,143,601,3547557.000,340190.000,65056.000,44845.000,55616.000,0.192,0.875,0.084,0.016,0.011,0.014
1,2012,2,2012-02-01,2012-02-29,29,696,3885254.000,5582.261,273,423,3386822.000,336345.000,63280.000,44627.000,54180.000,0.392,0.872,0.087,0.016,0.011,0.014
2,2012,3,2012-03-01,2012-03-30,30,719,4137599.000,5754.658,260,459,3568962.000,366898.000,74696.000,48150.000,78893.000,0.362,0.863,0.089,0.018,0.012,0.019
3,2012,4,2012-04-01,2012-04-30,29,696,3633084.000,5219.948,247,449,3115991.000,320089.000,67824.000,42285.000,86895.000,0.355,0.858,0.088,0.019,0.012,0.024
4,2012,5,2012-05-01,2012-05-31,31,744,4289819.000,5765.886,286,458,3657163.000,382935.000,77186.000,49772.000,122763.000,0.384,0.853,0.089,0.018,0.012,0.029


## Merge Area C Exposure Into Ring-Level Crashes

Area C is inside the Mura Spagnole. To avoid pretending it measures all Milan traffic, the merged file marks exposure as applicable only for the two central ring labels inside the Mura Spagnole.


In [8]:
ring_path = processed_dir / "milan_crashes_monthly_city_ring_cleaned.csv"
if ring_path.exists():
    rings = pd.read_csv(ring_path, parse_dates=["month_start"])
    rings["inside_mura_spagnole"] = rings["Cerchia"].isin(CENTRAL_RINGS_INSIDE_MURA_SPAGNOLE)
    exposure_cols = [
        "year", "month", "total_accesses", "mean_hourly_accesses", "days_observed", "hourly_rows",
        "active_hours", "inactive_hours", "areac_active_hour_share",
        *category_cols,
        *[col.replace("accesses_", "share_") for col in category_cols],
    ]
    monthly_for_merge = area_c_monthly[exposure_cols].rename(columns={"year": "Anno", "month": "Mese"})
    ring_exposure = rings.merge(monthly_for_merge, on=["Anno", "Mese"], how="left")
    area_c_cols = [col for col in ring_exposure.columns if col in monthly_for_merge.columns and col not in ["Anno", "Mese"]]
    ring_exposure.loc[~ring_exposure["inside_mura_spagnole"], area_c_cols] = pd.NA

    for outcome, rate_name in [
        ("Incidenti", "incidents_per_100k_area_c_accesses"),
        ("Feriti", "injuries_per_100k_area_c_accesses"),
        ("Morti", "deaths_per_100k_area_c_accesses"),
    ]:
        ring_exposure[rate_name] = (
            ring_exposure[outcome] / ring_exposure["total_accesses"].where(ring_exposure["total_accesses"].notna()) * 100_000
        )

    central_exposure = ring_exposure[ring_exposure["inside_mura_spagnole"] & ring_exposure["total_accesses"].notna()].copy()
    print("Merged ring exposure shape:", ring_exposure.shape)
    print("Central rows with Area C exposure:", central_exposure.shape)
    display(central_exposure.head())
else:
    print(f"Skipping ring merge because {ring_path.relative_to(project_root)} does not exist")
    ring_exposure = pd.DataFrame()


Merged ring exposure shape: (1440, 28)
Central rows with Area C exposure: (112, 28)


,Anno,Mese,Cerchia,Incidenti,Morti,Feriti,month_start,inside_mura_spagnole,total_accesses,mean_hourly_accesses,days_observed,hourly_rows,active_hours,inactive_hours,areac_active_hour_share,accesses_persone,accesses_merci,accesses_bus,accesses_altro,accesses_nc,share_persone,share_merci,share_bus,share_altro,share_nc,incidents_per_100k_area_c_accesses,injuries_per_100k_area_c_accesses,deaths_per_100k_area_c_accesses
660,2012,1,Dalla Cerchia dei Navigli alle Mura Spagnole,68,0,83,2012-01-01,True,4053264.000,5447.935,31.000,744.000,143,601,0.192,3547557.000,340190.000,65056.000,44845.000,55616.000,0.875,0.084,0.016,0.011,0.014,1.678,2.048,0.000
663,2012,1,Entro la Cerchia dei Navigli,31,0,35,2012-01-01,True,4053264.000,5447.935,31.000,744.000,143,601,0.192,3547557.000,340190.000,65056.000,44845.000,55616.000,0.875,0.084,0.016,0.011,0.014,0.765,0.864,0.000
665,2012,2,Dalla Cerchia dei Navigli alle Mura Spagnole,36,0,41,2012-02-01,True,3885254.000,5582.261,29.000,696.000,273,423,0.392,3386822.000,336345.000,63280.000,44627.000,54180.000,0.872,0.087,0.016,0.011,0.014,0.927,1.055,0.000
668,2012,2,Entro la Cerchia dei Navigli,19,0,23,2012-02-01,True,3885254.000,5582.261,29.000,696.000,273,423,0.392,3386822.000,336345.000,63280.000,44627.000,54180.000,0.872,0.087,0.016,0.011,0.014,0.489,0.592,0.000
670,2012,3,Dalla Cerchia dei Navigli alle Mura Spagnole,67,0,80,2012-03-01,True,4137599.000,5754.658,30.000,719.000,260,459,0.362,3568962.000,366898.000,74696.000,48150.000,78893.000,0.863,0.089,0.018,0.012,0.019,1.619,1.933,0.000


## Save Processed Files


In [9]:
long_path = processed_dir / "area_c_hourly_vehicle_category.csv"
wide_path = processed_dir / "area_c_hourly_wide.csv"
monthly_path = processed_dir / "area_c_monthly_exposure.csv"
ring_exposure_path = processed_dir / "milan_crashes_city_ring_area_c_exposure.csv"

area_c_long.to_csv(long_path, index=False)
area_c_wide.to_csv(wide_path, index=False)
area_c_monthly.to_csv(monthly_path, index=False)
if not ring_exposure.empty:
    ring_exposure.to_csv(ring_exposure_path, index=False)

print(f"Saved {long_path.relative_to(project_root)} rows={len(area_c_long):,}")
print(f"Saved {wide_path.relative_to(project_root)} rows={len(area_c_wide):,}")
print(f"Saved {monthly_path.relative_to(project_root)} rows={len(area_c_monthly):,}")
if not ring_exposure.empty:
    print(f"Saved {ring_exposure_path.relative_to(project_root)} rows={len(ring_exposure):,}")


Saved data/processed/area_c_hourly_vehicle_category.csv rows=193,845
Saved data/processed/area_c_hourly_wide.csv rows=39,109
Saved data/processed/area_c_monthly_exposure.csv rows=56
Saved data/processed/milan_crashes_city_ring_area_c_exposure.csv rows=1,440


## Add Long-Run Daily Total Area C Exposure

The hourly vehicle-category family is useful for composition but stops at partial 2016. The portal also provides daily total Area C accesses for 2016-2018 and 2019-2024. This section builds the longer monthly exposure denominator used in the ring notebooks.

In [10]:

DAILY_SOURCES = {
    "2016_2018": {
        "page": "https://dati.comune.milano.it/dataset/792cfd91-6cea-40d3-bcc2-feda97c110e4",
        "csv_url": "https://dati.comune.milano.it/dataset/792cfd91-6cea-40d3-bcc2-feda97c110e4/resource/377d041a-889b-4016-98d6-b0678e9ba5fc/download/ds1085_ingressi_areac_precedenti.csv",
        "raw_path": raw_dir / "area_c_accessi_giornalieri_2016_2018.csv",
    },
    "2019_2023": {
        "page": "https://dati.comune.milano.it/dataset/8937eb87-2356-40ba-bd82-e0fabe38b598",
        "csv_url": "https://dati.comune.milano.it/dataset/8937eb87-2356-40ba-bd82-e0fabe38b598/resource/c2f46ef8-9ee8-4883-807d-93adeb1b9931/download/ingressi_areac_2019-2023_no_dic.csv",
        "raw_path": raw_dir / "area_c_accessi_giornalieri_2019_2023.csv",
    },
    "2024": {
        "page": "https://dati.comune.milano.it/dataset/8937eb87-2356-40ba-bd82-e0fabe38b598",
        "csv_url": "https://dati.comune.milano.it/dataset/8937eb87-2356-40ba-bd82-e0fabe38b598/resource/b25e13d8-7fcb-46e3-b1e9-ff81b18f5c84/download/ingressi_areac_2025-01-17.csv",
        "raw_path": raw_dir / "area_c_accessi_giornalieri_2024.csv",
    },
}

for key, meta in DAILY_SOURCES.items():
    if meta["raw_path"].exists():
        print(f"{key}: found {meta['raw_path'].relative_to(project_root)}")
    else:
        print(f"{key}: downloading from {meta['csv_url']}")
        response = requests.get(meta["csv_url"], timeout=60)
        response.raise_for_status()
        meta["raw_path"].write_bytes(response.content)
        print(f"{key}: saved {meta['raw_path'].relative_to(project_root)}")


2016_2018: downloading from https://dati.comune.milano.it/dataset/792cfd91-6cea-40d3-bcc2-feda97c110e4/resource/377d041a-889b-4016-98d6-b0678e9ba5fc/download/ds1085_ingressi_areac_precedenti.csv
2016_2018: saved data/raw/area_c_accessi_giornalieri_2016_2018.csv
2019_2023: downloading from https://dati.comune.milano.it/dataset/8937eb87-2356-40ba-bd82-e0fabe38b598/resource/c2f46ef8-9ee8-4883-807d-93adeb1b9931/download/ingressi_areac_2019-2023_no_dic.csv


2019_2023: saved data/raw/area_c_accessi_giornalieri_2019_2023.csv
2024: downloading from https://dati.comune.milano.it/dataset/8937eb87-2356-40ba-bd82-e0fabe38b598/resource/b25e13d8-7fcb-46e3-b1e9-ff81b18f5c84/download/ingressi_areac_2025-01-17.csv
2024: saved data/raw/area_c_accessi_giornalieri_2024.csv


In [11]:

def clean_daily_access_file(path, source_name):
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    print("=" * 80)
    print(source_name)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Dtypes:")
    print(df.dtypes)
    print("Missing values:")
    print(df.isna().sum())
    display(df.head())

    working = df.copy()
    working.columns = [snake_case(col) for col in working.columns]
    date_col = find_one(working.columns, ["data", "date", "giorno"])
    total_candidates = [
        col for col in working.columns
        if ("transiti" in col or "accessi" in col or "ingressi" in col) and pd.to_numeric(working[col], errors="coerce").notna().any()
    ]
    preferred = [col for col in total_candidates if "h24" in col or "giornalieri" in col]
    total_col = preferred[0] if preferred else total_candidates[0]

    out = working[[date_col, total_col]].rename(columns={date_col: "date", total_col: "total_accesses"})
    out["date"] = pd.to_datetime(out["date"], errors="coerce", utc=True).dt.tz_convert(None).dt.date
    out["total_accesses"] = pd.to_numeric(out["total_accesses"], errors="coerce")
    out["source"] = source_name
    out = out.dropna(subset=["date", "total_accesses"]).drop_duplicates(subset=["date"], keep="last")
    return out

cleaned_daily_sources = [
    clean_daily_access_file(meta["raw_path"], key)
    for key, meta in DAILY_SOURCES.items()
]
portal_daily = pd.concat(cleaned_daily_sources, ignore_index=True).drop_duplicates(subset=["date"], keep="last")

hourly_daily = (
    area_c_wide
    .groupby("date", as_index=False)
    .agg(
        total_accesses=("total_accesses", "sum"),
        accesses_persone=("accesses_persone", "sum"),
        accesses_merci=("accesses_merci", "sum"),
        accesses_bus=("accesses_bus", "sum"),
        accesses_altro=("accesses_altro", "sum"),
        accesses_nc=("accesses_nc", "sum"),
        hourly_rows=("hour", "size"),
    )
)
hourly_daily["source"] = "hourly_category_2012_2016"
hourly_daily["date"] = pd.to_datetime(hourly_daily["date"]).dt.date

portal_dates = set(portal_daily["date"])
hourly_only = hourly_daily[~hourly_daily["date"].isin(portal_dates)].copy()
for col in ["accesses_persone", "accesses_merci", "accesses_bus", "accesses_altro", "accesses_nc", "hourly_rows"]:
    if col not in portal_daily.columns:
        portal_daily[col] = pd.NA
area_c_daily = pd.concat([hourly_only, portal_daily[hourly_only.columns]], ignore_index=True, sort=False)
area_c_daily["date"] = pd.to_datetime(area_c_daily["date"])
area_c_daily = area_c_daily.sort_values("date").reset_index(drop=True)
area_c_daily["year"] = area_c_daily["date"].dt.year.astype("Int64")
area_c_daily["month"] = area_c_daily["date"].dt.month.astype("Int64")
area_c_daily["day"] = area_c_daily["date"].dt.day.astype("Int64")
area_c_daily["weekday"] = area_c_daily["date"].dt.weekday.astype("Int64")
area_c_daily["is_weekend"] = area_c_daily["weekday"].isin([5, 6])

print("Daily exposure shape:", area_c_daily.shape)
print("Daily exposure range:", area_c_daily["date"].min().date(), "to", area_c_daily["date"].max().date())
print("Missing values:")
print(area_c_daily.isna().sum())
display(area_c_daily.head())
display(area_c_daily.tail())


2016_2018
Shape: (850, 2)
Columns: ['data_giorno', 'numero_transiti_giornalieri']
Dtypes:
data_giorno                      str
numero_transiti_giornalieri    int64
dtype: object
Missing values:
data_giorno                    0
numero_transiti_giornalieri    0
dtype: int64


,data_giorno,numero_transiti_giornalieri
0,2018-12-31,75579
1,2018-12-30,80642
2,2018-12-29,92106
3,2018-12-28,102149
4,2018-12-27,108274


2019_2023
Shape: (1781, 2)
Columns: ['data_giorno', 'numero_transiti_giornalieri']
Dtypes:
data_giorno                      str
numero_transiti_giornalieri    int64
dtype: object
Missing values:
data_giorno                    0
numero_transiti_giornalieri    0
dtype: int64


,data_giorno,numero_transiti_giornalieri
0,2023-11-30 00:00:00+01:00,142785
1,2023-11-29 00:00:00+01:00,144304
2,2023-11-28 00:00:00+01:00,145418
3,2023-11-27 00:00:00+01:00,134512
4,2023-11-26 00:00:00+01:00,107518


2024
Shape: (305, 9)
Columns: ['data_giorno', 'giorno', 'totale_transiti_giornalieri_h24', 'totale_transiti_giornalieri_7.30_19.30_autoveicoli', 'totale_transiti_giornalieri_7.30_19.30_ciclomotori_motocicli', 'totale_transiti_giornalieri_00.00_7.29_autoveicoli', 'totale_transiti_giornalieri_00.00_7.29_ciclomotori_motocicli', 'totale_transiti_giornalieri_19.30_23.59_autoveicoli', 'totale_transiti_giornalieri_19.30_23.59_ciclomotori_motocicli']
Dtypes:
data_giorno                                                        str
giorno                                                             str
totale_transiti_giornalieri_h24                                  int64
totale_transiti_giornalieri_7.30_19.30_autoveicoli               int64
totale_transiti_giornalieri_7.30_19.30_ciclomotori_motocicli     int64
totale_transiti_giornalieri_00.00_7.29_autoveicoli               int64
totale_transiti_giornalieri_00.00_7.29_ciclomotori_motocicli     int64
totale_transiti_giornalieri_19.30_23.59_autoveic

,data_giorno,giorno,totale_transiti_giornalieri_h24,totale_transiti_giornalieri_7.30_19.30_autoveicoli,totale_transiti_giornalieri_7.30_19.30_ciclomotori_motocicli,totale_transiti_giornalieri_00.00_7.29_autoveicoli,totale_transiti_giornalieri_00.00_7.29_ciclomotori_motocicli,totale_transiti_giornalieri_19.30_23.59_autoveicoli,totale_transiti_giornalieri_19.30_23.59_ciclomotori_motocicli
0,2024-11-30 00:00:00+01:00,sabato,122931,75332,9712,13491,965,21745,1686
1,2024-11-29 00:00:00+01:00,venerdì,140370,78471,21552,14165,1061,22538,2583
2,2024-11-28 00:00:00+01:00,giovedì,139258,78786,22959,12768,1022,21233,2490
3,2024-11-27 00:00:00+01:00,mercoledì,134273,77267,21522,12017,954,20046,2467
4,2024-11-26 00:00:00+01:00,martedì,130119,79270,17258,11609,837,19130,2015


Daily exposure shape: (4549, 14)
Daily exposure range: 2012-01-01 to 2024-11-29
Missing values:
date                   0
total_accesses         0
accesses_persone    2935
accesses_merci      2935
accesses_bus        2935
accesses_altro      2935
accesses_nc         2935
hourly_rows         2935
source                 0
year                   0
month                  0
day                    0
weekday                0
is_weekend             0
dtype: int64


,date,total_accesses,accesses_persone,accesses_merci,accesses_bus,accesses_altro,accesses_nc,hourly_rows,source,year,month,day,weekday,is_weekend
0,2012-01-01,77124.000,72390.000,1699.000,1414.000,347.000,1274.000,24,hourly_category_2012_2016,2012,1,1,6,True
1,2012-01-02,92461.000,79217.000,9199.000,1654.000,1150.000,1241.000,24,hourly_category_2012_2016,2012,1,2,0,False
2,2012-01-03,109458.000,93285.000,11323.000,1803.000,1532.000,1515.000,24,hourly_category_2012_2016,2012,1,3,1,False
3,2012-01-04,116050.000,99545.000,11716.000,1763.000,1493.000,1533.000,24,hourly_category_2012_2016,2012,1,4,2,False
4,2012-01-05,128023.000,111274.000,11727.000,1739.000,1630.000,1653.000,24,hourly_category_2012_2016,2012,1,5,3,False


,date,total_accesses,accesses_persone,accesses_merci,accesses_bus,accesses_altro,accesses_nc,hourly_rows,source,year,month,day,weekday,is_weekend
4544,2024-11-25,130119.000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024,2024,11,25,0,False
4545,2024-11-26,134273.000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024,2024,11,26,1,False
4546,2024-11-27,139258.000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024,2024,11,27,2,False
4547,2024-11-28,140370.000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024,2024,11,28,3,False
4548,2024-11-29,122931.000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024,2024,11,29,4,False


In [12]:

category_cols = ["accesses_persone", "accesses_merci", "accesses_bus", "accesses_altro", "accesses_nc"]
monthly_base = (
    area_c_daily
    .groupby(["year", "month"], as_index=False)
    .agg(
        month_start=("date", "min"),
        month_end=("date", "max"),
        days_observed=("date", "nunique"),
        total_accesses=("total_accesses", "sum"),
        mean_daily_accesses=("total_accesses", "mean"),
        source_days_hourly_category=("source", lambda s: int((s == "hourly_category_2012_2016").sum())),
        source_days_daily_total=("source", lambda s: int((s != "hourly_category_2012_2016").sum())),
    )
)
monthly_categories = area_c_daily.groupby(["year", "month"], as_index=False)[category_cols].sum(min_count=1)
area_c_monthly = monthly_base.merge(monthly_categories, on=["year", "month"], how="left")
for col in category_cols:
    area_c_monthly[col.replace("accesses_", "share_")] = area_c_monthly[col] / area_c_monthly["total_accesses"].where(area_c_monthly["total_accesses"] != 0)
area_c_monthly["month_start"] = pd.to_datetime(area_c_monthly["month_start"]).dt.to_period("M").dt.to_timestamp()
area_c_monthly["month_end"] = pd.to_datetime(area_c_monthly["month_end"])

print("Long-run monthly exposure shape:", area_c_monthly.shape)
print("Long-run monthly date range:", area_c_monthly["month_start"].min().date(), "to", area_c_monthly["month_start"].max().date())
display(area_c_monthly.groupby("year").agg(months=("month", "nunique"), days=("days_observed", "sum"), accesses=("total_accesses", "sum")))


Long-run monthly exposure shape: (153, 19)
Long-run monthly date range: 2012-01-01 to 2024-11-01


,months,days,accesses
year,,,
2012,12,364,46601227.000
2013,12,365,47013852.000
2014,12,349,46124857.000
2015,11,313,39825727.000
2016,12,345,28501214.000
2017,12,365,51726569.000
2018,12,363,50438550.000
2019,12,365,50897956.000
2020,12,366,35392120.000


In [13]:

ring_path = processed_dir / "milan_crashes_monthly_city_ring_cleaned.csv"
if ring_path.exists():
    rings = pd.read_csv(ring_path, parse_dates=["month_start"])
    rings["inside_mura_spagnole"] = rings["Cerchia"].isin(CENTRAL_RINGS_INSIDE_MURA_SPAGNOLE)
    exposure_cols = [
        "year", "month", "total_accesses", "mean_daily_accesses", "days_observed",
        "source_days_hourly_category", "source_days_daily_total",
        *category_cols,
        *[col.replace("accesses_", "share_") for col in category_cols],
    ]
    monthly_for_merge = area_c_monthly[exposure_cols].rename(columns={"year": "Anno", "month": "Mese"})
    ring_exposure = rings.merge(monthly_for_merge, on=["Anno", "Mese"], how="left")
    area_c_cols = [col for col in monthly_for_merge.columns if col not in ["Anno", "Mese"]]
    ring_exposure.loc[~ring_exposure["inside_mura_spagnole"], area_c_cols] = pd.NA

    for outcome, rate_name in [
        ("Incidenti", "incidents_per_100k_area_c_accesses"),
        ("Feriti", "injuries_per_100k_area_c_accesses"),
        ("Morti", "deaths_per_100k_area_c_accesses"),
    ]:
        ring_exposure[rate_name] = ring_exposure[outcome] / ring_exposure["total_accesses"].where(ring_exposure["total_accesses"].notna()) * 100_000

    central_exposure = ring_exposure[ring_exposure["inside_mura_spagnole"] & ring_exposure["total_accesses"].notna()].copy()
    print("Long-run merged ring exposure shape:", ring_exposure.shape)
    print("Central rows with Area C exposure:", central_exposure.shape)
    print("Coverage:", central_exposure["month_start"].min().date(), "to", central_exposure["month_start"].max().date())
    display(central_exposure.head())
else:
    print(f"Skipping ring merge because {ring_path.relative_to(project_root)} does not exist")
    ring_exposure = pd.DataFrame()


Long-run merged ring exposure shape: (1440, 26)
Central rows with Area C exposure: (306, 26)
Coverage: 2012-01-01 to 2024-11-01


,Anno,Mese,Cerchia,Incidenti,Morti,Feriti,month_start,inside_mura_spagnole,total_accesses,mean_daily_accesses,days_observed,source_days_hourly_category,source_days_daily_total,accesses_persone,accesses_merci,accesses_bus,accesses_altro,accesses_nc,share_persone,share_merci,share_bus,share_altro,share_nc,incidents_per_100k_area_c_accesses,injuries_per_100k_area_c_accesses,deaths_per_100k_area_c_accesses
660,2012,1,Dalla Cerchia dei Navigli alle Mura Spagnole,68,0,83,2012-01-01,True,4053264.000,130750.452,31.000,31.000,0.000,3547557.000,340190.000,65056.000,44845.000,55616.000,0.875,0.084,0.016,0.011,0.014,1.678,2.048,0.000
663,2012,1,Entro la Cerchia dei Navigli,31,0,35,2012-01-01,True,4053264.000,130750.452,31.000,31.000,0.000,3547557.000,340190.000,65056.000,44845.000,55616.000,0.875,0.084,0.016,0.011,0.014,0.765,0.864,0.000
665,2012,2,Dalla Cerchia dei Navigli alle Mura Spagnole,36,0,41,2012-02-01,True,3885254.000,133974.276,29.000,29.000,0.000,3386822.000,336345.000,63280.000,44627.000,54180.000,0.872,0.087,0.016,0.011,0.014,0.927,1.055,0.000
668,2012,2,Entro la Cerchia dei Navigli,19,0,23,2012-02-01,True,3885254.000,133974.276,29.000,29.000,0.000,3386822.000,336345.000,63280.000,44627.000,54180.000,0.872,0.087,0.016,0.011,0.014,0.489,0.592,0.000
670,2012,3,Dalla Cerchia dei Navigli alle Mura Spagnole,67,0,80,2012-03-01,True,4137599.000,137919.967,30.000,30.000,0.000,3568962.000,366898.000,74696.000,48150.000,78893.000,0.863,0.089,0.018,0.012,0.019,1.619,1.933,0.000


In [14]:

daily_path = processed_dir / "area_c_daily_total_accesses.csv"
monthly_path = processed_dir / "area_c_monthly_exposure.csv"
ring_exposure_path = processed_dir / "milan_crashes_city_ring_area_c_exposure.csv"

area_c_daily.to_csv(daily_path, index=False)
area_c_monthly.to_csv(monthly_path, index=False)
if not ring_exposure.empty:
    ring_exposure.to_csv(ring_exposure_path, index=False)

print(f"Saved {daily_path.relative_to(project_root)} rows={len(area_c_daily):,}")
print(f"Saved {monthly_path.relative_to(project_root)} rows={len(area_c_monthly):,}")
if not ring_exposure.empty:
    print(f"Saved {ring_exposure_path.relative_to(project_root)} rows={len(ring_exposure):,}")


Saved data/processed/area_c_daily_total_accesses.csv rows=4,549
Saved data/processed/area_c_monthly_exposure.csv rows=153
Saved data/processed/milan_crashes_city_ring_area_c_exposure.csv rows=1,440
